<a href="https://colab.research.google.com/github/RobertoSonzini/NAML_project/blob/LSMT_implementation/LSTM_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def dsigmoid_from_output(s):
    # s = sigmoid(x)
    return s * (1 - s)

def dtanh_from_output(t):
    # t = tanh(x)
    return 1 - t ** 2

In [ ]:
class LSTM:
  '''
    input_size: numero di feature del singolo sample
    hidden_size: numero di celle di memoria. In pratica si tratta di specificare la grandezza del layer che contiene le celle LSTM.
    output_size: numero di valori di output.
  '''
  def __init__(self, input_size, hidden_size, output_size):
    self.input_size = input_size
    self.hidden_size = hidden_size
    self.output_size = output_size

    # Output layer
    self.V = np.random.uniform(-0.1, 0.1, (output_size, hidden_size))
    self.b_out = np.zeros((output_size, 1))

    # Forget gate
    self.U_f = np.random.uniform(-0.1, 0.1, (self.hidden_size, self.input_size))
    self.W_f = np.random.uniform(-0.1, 0.1, (self.hidden_size, self.hidden_size))
    self.b_f = np.ones((hidden_size, 1)) # aiutiamo a ricordare all'inizio settando il valore a 1

    # Input gate
    self.U_i = np.random.uniform(-0.1, 0.1, (hidden_size, input_size))
    self.W_i = np.random.uniform(-0.1, 0.1, (hidden_size, hidden_size))
    self.b_i = np.zeros((hidden_size, 1))

    # External input gate (Candidate)
    self.U_g = np.random.uniform(-0.1, 0.1, (hidden_size, input_size))
    self.W_g = np.random.uniform(-0.1, 0.1, (hidden_size, hidden_size))
    self.b_g = np.zeros((hidden_size, 1))

    # Output gate
    self.U_o = np.random.uniform(-0.1, 0.1, (hidden_size, input_size))
    self.W_o = np.random.uniform(-0.1, 0.1, (hidden_size, hidden_size))
    self.b_o = np.zeros((hidden_size, 1))

    # Initial states
    self.h = np.zeros((hidden_size, 1))
    self.s = np.zeros((hidden_size, 1))

  def forward_step(self, x_t):
    i_t = sigmoid(np.dot(self.U_i, x_t) + np.dot(self.W_i, self.h) + self.b_i)
    f_t = sigmoid(np.dot(self.U_f, x_t) + np.dot(self.W_f, self.h) + self.b_f)
    g_t = np.tanh(np.dot(self.U_g, x_t) + np.dot(self.W_g, self.h) + self.b_g)
    o_t = sigmoid(np.dot(self.U_o, x_t) + np.dot(self.W_o, self.h) + self.b_o)

    self.s = f_t * self.s + i_t * g_t
    self.h = o_t * np.tanh(self.s)

    return self.h, self.s

  def forward(self, X):
    """
    X: array di dimensione (seq_len, input_size)
    Ritorna:
        Y_seq: array di dimensione (seq_len, output_size)
    """
    Y_seq = []

    for t in range(X.shape[0]):
        x_t = X[t].reshape(-1, 1)
        h_t, _ = self.forward_step(x_t)

        # Output layer
        y_t = np.dot(self.V, h_t) + self.b_out
        Y_seq.append(y_t)

    # Converti in array (seq_len, output_size)
    Y_seq = np.hstack(Y_seq).T

    return Y_seq